In [ ]:
import pandas as pd
import numpy as np

# Load the data from Parquet
df = pd.read_parquet("../data/f1_master_lap_dataset.parquet")

C:\Users\siddh\AppData\Local\Temp\ipykernel_26744\2574579936.py:5: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/f1_master_lap_dataset.csv")


In [24]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("F1-Strategy-Engine")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

df = (
    spark.read.parquet("/mnt/data/f1_master_lap_dataset.parquet")
    .select(
        col("Year").alias("season"),
        col("Race").alias("race"),
        col("Driver").alias("driver_id"),
        col("Team").alias("team"),
        col("LapNumber").alias("lap"),
        col("LapTime").alias("lap_time"),
        col("Position").alias("position"),
        col("GapToLeader").alias("gap_to_leader"),
        col("IntervalToAhead").alias("interval_to_ahead"),
        col("Compound").alias("compound"),
        col("TyreLife").alias("tyre_age"),
        col("Stint").alias("stint"),
        col("HasPit").alias("is_pit_lap")
    )
    .filter(col("lap_time").isNotNull())
    .cache()
)

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [ ]:
MASTER_PATH = "../data/f1_master_lap_dataset.parquet"

df = spark.read.parquet(MASTER_PATH)

# Minimal sanity filtering
df = df.filter(
    (col("LapTime").isNotNull()) &
    (col("LapNumber").isNotNull()) &
    (col("DriverNumber").isNotNull())
)

df.printSchema()
df.count()

AnalysisException: [PARQUET_TYPE_ILLEGAL] Illegal Parquet type: INT64 (TIMESTAMP(NANOS,false)). SQLSTATE: 42846

In [2]:
df.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,PitDuration,RaceLap,QualiPosition,Q1Time,Q2Time,Q3Time,FinalPosition,Status,Points,FinalRaceTime
0,0 days 00:08:42.399000,HAM,44,0 days 00:01:34.233000,1.0,NaN,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:23.820000,...,0.0,1.0,1.0,0 days 00:01:22.824000,0 days 00:01:22.051000,0 days 00:01:21.164000,2.0,Finished,18.0,0 days 00:00:05.036000
1,0 days 00:08:43.640000,RAI,7,0 days 00:01:35.474000,1.0,NaN,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:23.858000,...,0.0,1.0,2.0,0 days 00:01:23.096000,0 days 00:01:22.507000,0 days 00:01:21.828000,3.0,Finished,15.0,0 days 00:00:06.309000
2,0 days 00:08:44.360000,VET,5,0 days 00:01:36.194000,1.0,NaN,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:24.142000,...,0.0,1.0,3.0,0 days 00:01:23.348000,0 days 00:01:21.944000,0 days 00:01:21.838000,1.0,Finished,25.0,0 days 01:29:33.283000
3,0 days 00:08:45.273000,MAG,20,0 days 00:01:37.107000,1.0,NaN,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:24.172000,...,0.0,1.0,6.0,0 days 00:01:23.909000,0 days 00:01:23.300000,0 days 00:01:23.187000,17.0,Wheel,0.0,NaN
4,0 days 00:08:45.850000,VER,33,0 days 00:01:37.684000,1.0,NaN,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:24.474000,...,0.0,1.0,4.0,0 days 00:01:23.483000,0 days 00:01:22.416000,0 days 00:01:21.879000,6.0,Finished,8.0,0 days 00:00:28.945000


In [3]:
df.columns

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate', 'AirTemp_C', 'Humidity_pct',
       'Pressure', 'Rainfall_mm', 'TrackTemp_C', 'WindDirection',
       'WindSpeed_kmh', 'Year', 'Race', 'Circuit', 'Location', 'Country',
       'SessionDate', 'TeamName', 'TeamColor', 'BroadcastName', 'GapToAhead',
       'DeltaToLeader', 'DeltaToAverage', 'HasPit', 'PitDuration', 'RaceLap',
       'QualiPosition', 'Q1Time', 'Q2Time', 'Q3Time', 'FinalPosition',
       'Status', 'Points', 'FinalRaceTime'],
      dtype='object')

In [4]:
columns_to_drop = ["Time", "Driver", "DriverNumber", "PitInTime", "PitOutTime", "Sector1SessionTime", "Sector2SessionTime", "Sector3SessionTime", "Team", "LapStartTime", "LapStartDate", "Position", "Deleted", "DeletedReason", "FastF1Generated", "IsAccurate", "Pressure", "Rainfall_mm", "WindDirection", "Race", "Circuit", "Location", "Country", "SessionDate", "TeamColor", "BroadcastName", "FinalPosition", "Status", "Points", "FinalRaceTime"]
df = df.drop(columns=columns_to_drop)


In [6]:
df = df[(df["Stint"].notna()) & (df["LapTime"] != "")]

In [7]:
df["TrackStatus"] = df["TrackStatus"].fillna(1)

In [9]:
timedelta_cols = [
    'LapTime',
    'Sector1Time', 'Sector2Time', 'Sector3Time',
    'GapToAhead',
    'DeltaToLeader',
    'DeltaToAverage',
    'PitDuration',
    'Q1Time', 'Q2Time', 'Q3Time'
]

for col in timedelta_cols:
    df[col] = pd.to_timedelta(df[col], errors='coerce')
    df[col] = df[col].dt.total_seconds()


In [11]:
# Boolean to integer

bool_cols = ['IsPersonalBest', 'FreshTyre', 'HasPit']

df[bool_cols] = (
    df[bool_cols]
    .fillna(False) 
    .astype(int)     
)

C:\Users\siddh\AppData\Local\Temp\ipykernel_26744\1455532994.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


In [12]:
#One hot encoding

df = pd.get_dummies(
    df,
    columns=['Compound'],
    prefix=['Compound'],
    drop_first=False
)

In [14]:
df = df.fillna(0)

In [16]:
#df.to_csv("../data/cleaned_data.csv", index=False)
#print("Data saved to new csv.")